# Nome: Alexandre Rodrigues Santarossa
## SuperComputação - Aula 14

In [2]:
%%writefile pi_monte_carlo_sequencial.cpp
#include <iostream>
#include <cmath>
#include <cstdlib>
#include <ctime>
#include <chrono>

int main() {
    const int N = 100000;
    int pontos_no_circulo = 0;

    srand(static_cast<unsigned>(time(nullptr)));

    auto inicio = std::chrono::high_resolution_clock::now();


    for (int i = 0; i < N; ++i) {
        double x = static_cast<double>(rand()) / RAND_MAX;
        double y = static_cast<double>(rand()) / RAND_MAX;

        if ((x * x + y * y) <= 1.0) {
            ++pontos_no_circulo;
        }
    }

    double pi_estimado = 4.0 * pontos_no_circulo / N;

    auto fim = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> duracao = fim - inicio;

    std::cout << "Número de pontos dentro do círculo: " << pontos_no_circulo << std::endl;
    std::cout << "Estimativa de Pi: " << pi_estimado << std::endl;
    std::cout << "Tempo de execução: " << duracao.count() << " segundos" << std::endl;

    return 0;
}

Overwriting pi_monte_carlo_sequencial.cpp


In [3]:
!g++ pi_monte_carlo_sequencial.cpp -o pi_seq
!./pi_seq

Número de pontos dentro do círculo: 78522
Estimativa de Pi: 3.14088
Tempo de execução: 0.00415521 segundos


In [4]:
%%writefile pi_monte_carlo_paralelo.cpp
#include <iostream>
#include <cmath>
#include <cstdlib>
#include <ctime>
#include <chrono>
#include <omp.h>
int main() {
    const int N = 100000;
    int contador_circulo = 0;

    srand(static_cast<unsigned>(time(nullptr)));

    auto inicio = std::chrono::high_resolution_clock::now();

    #pragma omp parallel for reduction(+:contador_circulo)
    for (int i = 0; i < N; ++i) {
        double x = static_cast<double>(rand()) / RAND_MAX;
        double y = static_cast<double>(rand()) / RAND_MAX;
        if (x * x + y * y <= 1.0) {
            ++contador_circulo;
        }
    }

    double pi_estimado = 4.0 * contador_circulo / N;

    auto fim = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> duracao = fim - inicio;

    std::cout << "Número de pontos dentro do círculo: " << contador_circulo << std::endl;
    std::cout << "Estimativa de Pi: " << pi_estimado << std::endl;
    std::cout << "Tempo de execução (versão paralela): " << duracao.count() << " segundos" << std::endl;

    return 0;
}

Writing pi_monte_carlo_paralelo.cpp


In [5]:
!g++ -fopenmp pi_monte_carlo_paralelo.cpp -o pi_par
!./pi_par

Número de pontos dentro do círculo: 78361
Estimativa de Pi: 3.13444
Tempo de execução (versão paralela): 0.014815 segundos


In [6]:
%%writefile pi_monte_carlo_paralelo_melhorado.cpp
#include <iostream>
#include <cmath>
#include <random>
#include <chrono>
#include <omp.h>

int main() {
    const int N = 100000;
    int contador_circulo = 0;

    auto inicio = std::chrono::high_resolution_clock::now();

    #pragma omp parallel reduction(+:contador_circulo)
    {
        std::random_device rd;
        std::mt19937 gen(rd() + omp_get_thread_num());
        std::uniform_real_distribution<> dis(0.0, 1.0);

        #pragma omp for
        for (int i = 0; i < N; ++i) {
            double x = dis(gen);
            double y = dis(gen);

            if (x * x + y * y <= 1.0) {
                ++contador_circulo;
            }
        }
    }

    double pi_estimado = 4.0 * contador_circulo / N;

    auto fim = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> duracao = fim - inicio;

    std::cout << "Número de pontos dentro do círculo: " << contador_circulo << std::endl;
    std::cout << "Estimativa de Pi (versão paralela melhorada): " << pi_estimado << std::endl;
    std::cout << "Tempo de execução (versão paralela melhorada): " << duracao.count() << " segundos" << std::endl;

    return 0;
}

Writing pi_monte_carlo_paralelo_melhorado.cpp


In [7]:
!g++ -fopenmp pi_monte_carlo_paralelo_melhorado.cpp -o pi_par_melh
!./pi_par_melh

Número de pontos dentro do círculo: 78520
Estimativa de Pi (versão paralela melhorada): 3.1408
Tempo de execução (versão paralela melhorada): 0.0417105 segundos


# Resultados do Cluster:

### *Sequencial* :	Valor de pi estimado de 3.14656	 com tempo de execuçção de 0.00466573s

### *Paralelizada* :	Valor de pi estimado de 3.14912	 com tempo de execuçção de 0.0181428s

### *Paralelizada 2a Versão* :	Valor de pi estimado de 3.15016	 com tempo de execuçção de 0.0129763s


# Reflexões e Conclusão

1. Houve uma melhoria significativa no tempo de execução entre a versão sequencial e as versões paralelas?

  Sim, principalmente na  segunda versão paralela, onde a distribuição do job entre threads reduziu o tempo de execução em comparação com a versão sequencial.
2. A estimativa de pi permaneceu precisa em todas as versões?
  Sim, a estimativa de pi manteve-se consistente e próxima do valor real em todas as versões implementadas.

3. Quais foram os maiores desafios ao paralelizar o algoritmo, especialmente em relação aos números aleatórios?
  O maior desafio foi gerar números aleatórios sem conflitos dentro de um ambiente paralelo, onde foi introduzido um gerador para cada thread utilizada.

4. O uso de threads trouxe benefícios claros para este problema específico?
  Sim, o uso de threads melhorou o desempenho ao reduzir o tempo de execução, mostrando os benefícios da paralelização do algoritmo.
